# Ноутбук 04 — Эконометрические модели: SARIMA и Holt-Winters
**Подраздел 3.2.5 ПЗ** — SARIMA идентификация и оценка  
**Подраздел 3.2.6 ПЗ** — Holt-Winters подбор параметров сглаживания  
**Подраздел 3.3 ПЗ** — результаты

Область применения: агрегированный недельный ряд суммарных продаж  
(сумма по всем 54 магазинам и 33 семействам товаров).  
Обоснование: SARIMA и Holt-Winters требуют единственного временного ряда;  
обучение 1782 отдельных моделей нецелесообразно в рамках академического сравнения.

Артефакты:
- `models/saved/sarima_agg.pkl`  
- `models/saved/holtwinters_agg.pkl`  
- `reports/tables/table_3_metrics_econometric.csv`  
- `reports/figures/fig_3_forecast_sarima.png`  
- `reports/figures/fig_3_forecast_hw.png`  
- `reports/figures/fig_3_sarima_residuals.png`


In [ ]:
import sys, warnings, pickle
sys.path.insert(0, "..")
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams["figure.dpi"] = 120

from src.config import (
    DATA_INT, DATA_PROC, MODELS_DIR, TABLES, FIGURES,
    TARGET, DATE_COL, STORE_COL, FAMILY_COL,
    FORECAST_HORIZONS, TRAIN_CUTOFF, SEASONAL_PERIOD,
)
from src.evaluation.metrics import compute_metrics, metrics_table
from src.models.sarima_model import (
    fit_sarima, sarima_forecast, sarima_residual_diagnostics,
    SARIMA_ORDER, SARIMA_SEASONAL_ORDER,
)
from src.models.ets_model import fit_holtwinters, holtwinters_forecast, holtwinters_params

print("Импорты выполнены.")
print(f"SARIMA спецификация: {SARIMA_ORDER} × {SARIMA_SEASONAL_ORDER}")
print(f"Сезонный период S = {SEASONAL_PERIOD}")


## Ячейка 1 — Загрузка и подготовка агрегированного ряда

Агрегированный недельный ряд суммарных продаж загружается из  
`data/interim/weekly_sales.parquet` (артефакт ноутбука 01, ячейка 7).  
Применяется log1p-преобразование для согласованности с ML-моделями.


In [ ]:
# Загрузка агрегированного ряда
weekly_raw = pd.read_parquet(DATA_INT / "weekly_sales.parquet")
# Убеждаемся в правильном формате Series с частотой W-MON
if isinstance(weekly_raw, pd.DataFrame):
    weekly_raw = weekly_raw.iloc[:, 0]
weekly_raw = weekly_raw.sort_index()
weekly_raw.index = pd.to_datetime(weekly_raw.index)
weekly_raw = weekly_raw.asfreq("W-MON", method="ffill")

print(f"Агрегированный ряд: {len(weekly_raw)} недель")
print(f"Диапазон: {weekly_raw.index.min().date()} — {weekly_raw.index.max().date()}")
print(f"Min={weekly_raw.min():.1f}, Max={weekly_raw.max():.1f}, Mean={weekly_raw.mean():.1f}")

# log1p-преобразование
weekly_log = np.log1p(weekly_raw)
print(f"\nПосле log1p: mean={weekly_log.mean():.4f}, std={weekly_log.std():.4f}")

# Разбивка на train/test
cutoff = pd.Timestamp(TRAIN_CUTOFF)
train_series = weekly_log[weekly_log.index < cutoff]
test_series  = weekly_log[weekly_log.index >= cutoff]
print(f"\nTrain: {len(train_series)} недель | Test: {len(test_series)} недель")

# Визуализация
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(train_series.index, np.expm1(train_series), label="Train (факт)", color="steelblue")
ax.plot(test_series.index,  np.expm1(test_series),  label="Test (факт)",  color="tomato")
ax.axvline(cutoff, color="black", linestyle="--", linewidth=1, label="Граница train/test")
ax.set_title("Агрегированный недельный ряд суммарных продаж")
ax.set_ylabel("Суммарные продажи, ед.")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES / "fig_3_weekly_agg_split.png", dpi=120)
plt.show()


## Ячейка 2 — SARIMA: обучение и диагностика

**Спецификация SARIMA(1,1,0)(0,1,1)[52]** выведена в пункте 2.3.4 ПЗ:
- d=1: ADF p=0,4893 (Таблица 2.2), нестационарность подтверждена.
- p=1: единственный значимый лаг ЧАКФ исходного ряда (Рисунок 2.9).
- D=1: нестационарность сезонного компонента STL (Рисунок 2.4).
- Q=1: пик АКФ на лаге 52, без значимого пика ЧАКФ на лаге 52 (Таблица 2.3).


In [ ]:
print(f"Обучение SARIMA{SARIMA_ORDER}×{SARIMA_SEASONAL_ORDER}...")
print("(Может занять 1-5 минут для S=52)")

sarima_result = fit_sarima(train_series, SARIMA_ORDER, SARIMA_SEASONAL_ORDER)
print("SARIMA обучен.")
print(f"AIC = {sarima_result.aic:.2f}")
print(f"BIC = {sarima_result.bic:.2f}")
print(f"\nКраткая сводка:")
print(sarima_result.summary().tables[1])


In [ ]:
# Диагностика остатков SARIMA
diag = sarima_residual_diagnostics(sarima_result)
print("Тест Льюнга-Бокса (H₀: нет автокорреляции остатков):")
print(diag["ljung_box"].to_string())
print(f"\nТест нормальности Жарка-Бера: p = {diag['jarque_bera_p']:.4f}")

# График остатков SARIMA
fig = sarima_result.plot_diagnostics(figsize=(14, 8))
fig.suptitle("Рисунок 3.8 — Диагностика остатков SARIMA(1,1,0)(0,1,1)[52]")
plt.tight_layout()
plt.savefig(FIGURES / "fig_3_sarima_residuals.png", dpi=120)
plt.show()


## Ячейка 3 — SARIMA: прогноз по горизонтам

Прогноз выполняется методом rolling evaluation (скользящая оценка):  
для каждой тестовой недели модель обновляется с новым наблюдением (append),  
затем делает прогноз на h шагов вперёд.

Это обеспечивает честную сравнительную оценку с ML-моделями.


In [ ]:
results_sarima = {}

for h in FORECAST_HORIZONS:
    print(f"\nSARIMA | Горизонт h = {h} нед.")

    actuals_log = []
    preds_log   = []

    # Скользящая оценка (rolling evaluation)
    # Для каждой позиции в test_series делаем прогноз на h шагов
    # и берём только h-й шаг
    history = train_series.copy()
    test_vals = test_series.values

    for i in range(len(test_vals)):
        if i + h > len(test_vals):
            break
        # Обновляем модель с наблюдением i (если i > 0)
        if i > 0:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                # Добавляем наблюдение к истории
                new_obs = pd.Series([test_vals[i - 1]], index=[test_series.index[i - 1]])
                history = pd.concat([history, new_obs])

        # Прогноз на h шагов
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            model_upd = fit_sarima(history, SARIMA_ORDER, SARIMA_SEASONAL_ORDER)
            fc = sarima_forecast(model_upd, steps=h)

        actuals_log.append(test_vals[i + h - 1])
        preds_log.append(fc[-1])  # берём h-й шаг прогноза

        if i % 3 == 0:
            print(f"  Шаг {i}/{len(test_vals)}: actual={np.expm1(test_vals[i+h-1]):.1f}, "
                  f"pred={np.expm1(fc[-1]):.1f}")

    actuals_log = np.array(actuals_log)
    preds_log   = np.array(preds_log)
    metrics_s   = compute_metrics(actuals_log, preds_log, log_scale=True)
    results_sarima[h] = metrics_s
    print(f"  SARIMA → RMSE={metrics_s['RMSE']:.2f}, MAE={metrics_s['MAE']:.2f}, MAPE={metrics_s['MAPE']:.2f}%")

print("\n[OK] SARIMA rolling evaluation завершена.")


## Ячейка 4 — Holt-Winters: обучение и прогноз

In [ ]:
print(f"Обучение Holt-Winters (ETS Add,Add,Add, S={SEASONAL_PERIOD})...")
hw_result = fit_holtwinters(train_series, seasonal_periods=SEASONAL_PERIOD)
print("Holt-Winters обучен.")
hw_p = holtwinters_params(hw_result)
print(f"  alpha = {hw_p['alpha']}  (сглаживание уровня)")
print(f"  beta  = {hw_p['beta']}   (сглаживание тренда)")
print(f"  gamma = {hw_p['gamma']}  (сглаживание сезонности)")
print(f"  SSE   = {hw_p['sse']:.2f}")
print(f"  AIC   = {hw_p['aic']:.2f}")

results_hw = {}
for h in FORECAST_HORIZONS:
    print(f"\nHolt-Winters | Горизонт h = {h} нед.")
    actuals_log = []
    preds_log   = []
    history = train_series.copy()
    test_vals = test_series.values

    for i in range(len(test_vals)):
        if i + h > len(test_vals):
            break
        if i > 0:
            new_obs = pd.Series([test_vals[i - 1]], index=[test_series.index[i - 1]])
            history = pd.concat([history, new_obs])
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                hw_upd = fit_holtwinters(history, seasonal_periods=SEASONAL_PERIOD)
        else:
            hw_upd = hw_result

        fc = holtwinters_forecast(hw_upd, steps=h)
        actuals_log.append(test_vals[i + h - 1])
        preds_log.append(fc[-1])

    actuals_log = np.array(actuals_log)
    preds_log   = np.array(preds_log)
    metrics_hw  = compute_metrics(actuals_log, preds_log, log_scale=True)
    results_hw[h] = metrics_hw
    print(f"  HW → RMSE={metrics_hw['RMSE']:.2f}, MAE={metrics_hw['MAE']:.2f}, MAPE={metrics_hw['MAPE']:.2f}%")

print("\n[OK] Holt-Winters завершён.")


## Ячейка 5 — Сравнительные графики прогноз vs факт

In [ ]:
# Статический прогноз (не rolling) для графика
h_plot = 1
sarima_fc_static = sarima_forecast(sarima_result, steps=len(test_series))
hw_fc_static     = holtwinters_forecast(hw_result, steps=len(test_series))

actual_orig = np.expm1(test_series.values)
sarima_orig = np.expm1(sarima_fc_static[:len(actual_orig)])
hw_orig     = np.expm1(hw_fc_static[:len(actual_orig)])

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# SARIMA
axes[0].plot(test_series.index, actual_orig,  label="Факт",          color="steelblue", linewidth=1.5)
axes[0].plot(test_series.index, sarima_orig,  label="SARIMA(1,1,0)(0,1,1)[52]",
             color="tomato", linestyle="--", linewidth=1.5)
axes[0].set_title("Рисунок 3.6 — SARIMA: прогноз vs факт")
axes[0].set_ylabel("Суммарные продажи, ед.")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Holt-Winters
axes[1].plot(test_series.index, actual_orig,  label="Факт",          color="steelblue", linewidth=1.5)
axes[1].plot(test_series.index, hw_orig,      label="Holt-Winters (ETS AAA)",
             color="green", linestyle="--", linewidth=1.5)
axes[1].set_title("Рисунок 3.7 — Holt-Winters: прогноз vs факт")
axes[1].set_ylabel("Суммарные продажи, ед.")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / "fig_3_forecast_sarima_hw.png", dpi=120)
plt.show()


## Ячейка 6 — Сохранение моделей и таблица метрик

In [ ]:
import pickle

with open(MODELS_DIR / "sarima_agg.pkl", "wb") as f:
    pickle.dump(sarima_result, f)
with open(MODELS_DIR / "holtwinters_agg.pkl", "wb") as f:
    pickle.dump(hw_result, f)

summary_econo = {"SARIMA": results_sarima, "HoltWinters": results_hw}
df_econo = metrics_table(summary_econo)
print("Таблица метрик эконометрических моделей:")
print(df_econo.to_string())
df_econo.to_csv(TABLES / "table_3_metrics_econometric.csv")
print("\nСохранено: reports/tables/table_3_metrics_econometric.csv")


In [ ]:
print("=" * 60)
print("Ноутбук 04 выполнен.")
print("=" * 60)
